# Tao: CUNP89T vs. Chang (ground truth)

Scores the **CUNP89T** Ruth/Jonah text against the **Chang** text, treating Chang as the reference. The verse alignment comes from `align_verses.py` (`benchmark_data/aligned-RUT-JON.json`), where a Chang verse range such as `1-2` is matched with the concatenation of the CUNP89T verses in that range.

Metrics follow `eval/flores/eval_nllb_gemini.py` and the other Tao notebook: corpus-level **chrF++** (`corpus_chrf(..., word_order=2)`) and **BLEU** (`corpus_bleu`), plus per-sentence chrF++.

The two texts are different translations of the same books, with different spelling (Naemey / Naomi, Poase / Poatse) and punctuation (Chang puts a space before punctuation and uses bracketed notes like `[10]`; CUNP89T uses curly quotes and `(10)`). So every score is reported twice: on the **raw** text and on a **normalized** text (lowercased, bracketed/parenthesised notes and punctuation removed). The normalized score shows how much of the gap is just formatting.

## 0. Setup

In [ ]:
%pip install -q sacrebleu pandas

In [ ]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from sacrebleu import corpus_chrf, corpus_bleu, sentence_chrf

pd.set_option("display.max_colwidth", 200)

DATA_FILE = Path("benchmark_data/aligned-RUT-JON.json")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

BOOK_NAMES = {"08": "Ruth", "32": "Jonah"}

## 1. Load the aligned data

Keys look like `BBCCCVVV-VVV` (book, chapter, first verse, last verse).

In [ ]:
with open(DATA_FILE, encoding="utf-8") as f:
    aligned = json.load(f)

rows = []
for key, pair in aligned.items():
    m = re.fullmatch(r"(\d{2})(\d{3})(\d{3})-(\d{3})", key)
    bb, ch, v1, v2 = m.groups()
    rows.append({
        "key": key,
        "book": BOOK_NAMES.get(bb, bb),
        "chapter": int(ch),
        "verses": f"{int(v1)}" if v1 == v2 else f"{int(v1)}-{int(v2)}",
        "ref": pair["Chang"],      # ground truth
        "hyp": pair["CUNP89T"],    # system under test
    })
df = pd.DataFrame(rows)

print(f"{len(df)} aligned units;", df["book"].value_counts().to_dict())
empty = df[(df["ref"] == "") | (df["hyp"] == "")]
print(f"units with an empty side: {len(empty)}")
df.head()

## 2. Normalization

Applied to both sides for the "normalized" scores: Unicode NFKC, lowercase, drop `[10]` / `(10)` / `(12,000,000)` style notes, drop all punctuation, collapse whitespace.

In [ ]:
NOTE_RE = re.compile(r"[\[(][\d,\s]+[\])]")

def normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).lower()
    text = NOTE_RE.sub(" ", text)
    text = "".join(" " if unicodedata.category(c).startswith("P") else c for c in text)
    return re.sub(r"\s+", " ", text).strip()

df["ref_norm"] = df["ref"].map(normalize)
df["hyp_norm"] = df["hyp"].map(normalize)
df[["ref", "ref_norm"]].iloc[[2, 40]]

## 3. Corpus-level scores

chrF++ with `word_order=2` and BLEU, exactly as in the other evaluation scripts.

In [ ]:
def corpus_scores(frame, hyp_col, ref_col):
    hyps, refs = frame[hyp_col].tolist(), [frame[ref_col].tolist()]
    return {
        "chrF++": corpus_chrf(hyps, refs, word_order=2).score,
        "BLEU": corpus_bleu(hyps, refs).score,
    }

summary = pd.DataFrame({
    "raw": corpus_scores(df, "hyp", "ref"),
    "normalized": corpus_scores(df, "hyp_norm", "ref_norm"),
}).T.round(2)
summary["n"] = len(df)
summary

## 4. By book and chapter

In [ ]:
def grouped(by):
    out = []
    for name, g in df.groupby(by, sort=False):
        raw = corpus_scores(g, "hyp", "ref")
        norm = corpus_scores(g, "hyp_norm", "ref_norm")
        out.append({
            "group": name if isinstance(name, str) else " ".join(map(str, name)),
            "n": len(g),
            "chrF++ raw": raw["chrF++"], "BLEU raw": raw["BLEU"],
            "chrF++ norm": norm["chrF++"], "BLEU norm": norm["BLEU"],
        })
    return pd.DataFrame(out).set_index("group").round(2)

display(grouped("book"))
display(grouped(["book", "chapter"]))

## 5. Per-sentence chrF++

Sentence-level chrF++ on the raw and normalized text. Low values mark verses where the two translations diverge the most.

In [ ]:
df["chrf_raw"] = [sentence_chrf(h, [r], word_order=2).score for h, r in zip(df["hyp"], df["ref"])]
df["chrf_norm"] = [sentence_chrf(h, [r], word_order=2).score for h, r in zip(df["hyp_norm"], df["ref_norm"])]
df["len_ratio"] = (df["hyp_norm"].str.split().str.len() / df["ref_norm"].str.split().str.len().clip(lower=1)).round(2)

display(df[["chrf_raw", "chrf_norm", "len_ratio"]].describe().round(2))
print("Normalization gain (mean sentence chrF++):",
      round(df["chrf_norm"].mean() - df["chrf_raw"].mean(), 2))

In [ ]:
cols = ["key", "chrf_norm", "len_ratio", "ref", "hyp"]
print("Lowest-scoring verses")
display(df.nsmallest(5, "chrf_norm")[cols])
print("Highest-scoring verses")
display(df.nlargest(5, "chrf_norm")[cols])

## 6. Save results

In [ ]:
df.drop(columns=["ref_norm", "hyp_norm"]).to_csv(RESULTS_DIR / "tao_cunp89t_vs_chang.csv", index=False)
summary.to_csv(RESULTS_DIR / "tao_cunp89t_vs_chang_summary.csv")
print("saved to", RESULTS_DIR.resolve())